In [1]:
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape
from shapely import wkt
import warnings
# filter all warnings
warnings.filterwarnings("ignore")

In [2]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)



HE SUBtituit els punts de les parades de fcg.metro amb els punts de les dades de tmb

In [3]:
with open('data/FGC/fcg_stops.json') as f:
    data = json.load(f)
fgc_stops = []
for stop in data['features']:
    properties = stop.get('properties', {})
    stop_id = properties.get('stop_id')
    stop_name = properties.get('stop_name')

    
    fgc_stops.append({
        'stop_id': stop_id,
        'stop_name': stop_name,
        'geometry': shape(stop['geometry'])

    })  
geo_df_fgc_stops = gpd.GeoDataFrame(fgc_stops, crs="EPSG:4326")
geo_df_fgc_stops.to_crs(epsg=25831, inplace=True)
neigh = neigh.to_crs(epsg=25831)
geo_df_fgc_stops = geo_df_fgc_stops[
    geo_df_fgc_stops.within(
        neigh.geometry.buffer(2500).union_all()
    )
]

geo_df_fgc_stops.to_crs(epsg=4326, inplace=True)
neigh.to_crs(epsg=4326, inplace=True)
geo_df_fgc_stops.sort_values(by='stop_id',inplace=True)
geo_df_fgc_stops.drop_duplicates(subset=['stop_name'], inplace=True,keep='first')
geo_df_fgc_stops['stop_name'].replace({
    'Barcelona - Plaça Catalunya': 'Catalunya',
    "L'Hospitalet Av. Carrilet": "Av. Carrilet",
    'Provença': 'Diagonal',
    'Barcelona - Plaça Espanya': 'Espanya',
}, inplace=True)

geo_df_fgc_stops = geo_df_fgc_stops[~geo_df_fgc_stops['stop_name'].isin(['Vallvidrera Inferior','Vallvidrera Superior'])]
geo_df_fgc_stops

,stop_id,stop_name,geometry
45,AL,Almeda,POINT (2.08525 41.35307)
4,BN,La Bonanova,POINT (2.13643 41.39782)
11,EP,El Putxet,POINT (2.13915 41.40583)
240,EU,Europa | Fira,POINT (2.12592 41.35756)
149,GO,Gornal,POINT (2.11743 41.35489)
203,GR,Gràcia,POINT (2.15266 41.3991)
235,IC,Ildefons Cerdà,POINT (2.13002 41.36091)
18,LF,La Floresta,POINT (2.07317 41.44487)
152,LH,Av. Carrilet,POINT (2.10265 41.35852)
218,LP,Les Planes,POINT (2.09162 41.4274)


# Solutions

In [4]:
solutions = geo_df_fgc_stops.copy()
already_in_metro = ['Catalunya', 'Diagonal', 'Espanya','Av. Carrilet','Europa | Fira']
solutions = solutions[~solutions['stop_name'].isin(already_in_metro)]
solutions.insert(2, 'stop_type', 'FGC')
solutions.rename(columns={'stop_name': 'name','stop_id': 'id'}, inplace=True)
solutions['id'] = 'SF' + '-' + solutions['id']

In [5]:
solutions.to_csv('Nodes/FGC-Sol.csv', index=False)

# FGC 

In [6]:
stops_map = {
    'Catalunya': ['L6', 'L7', 'S1'],
    'Diagonal': ['L6', 'L7', 'S1'],
    'Gràcia': ['L6', 'L7', 'S1'],
    'Sant Gervasi': ['L6', 'S1'],
    'Muntaner': ['L6', 'S1'],
    'La Bonanova': ['L6', 'S1'],
    'Les Tres Torres': ['L6', 'S1'],
    'Sarrià': ['L6', 'S1', 'L12'],
    'Pl. Molina': ['L7'],
    'Pàdua': ['L7'],
    'El Putxet': ['L7'],
    'Av. Tibidabo': ['L7'],
    'Espanya': ['L8'],
    'Magòria La Campana': ['L8'],
    'Ildefons Cerdà': ['L8'],
    'Europa | Fira': ['L8'],
    'Gornal': ['L8'],
    'Sant Josep': ['L8'],
    'Av. Carrilet': ['L8'],
    'Almeda': ['L8'],
    'Peu del Funicular': ['S1'],
    'Baixador de Vallvidrera': ['S1'],
    'Les Planes': ['S1'],
    'La Floresta': ['S1'],
    'Valldoreix': ['S1'],
    'Reina Elisenda': ['L12']
}

In [7]:
geo_df_fgc_stops['linia'] = geo_df_fgc_stops['stop_name'].map(stops_map)
geo_df_fgc_stops = geo_df_fgc_stops.explode('linia', ignore_index=True)
geo_df_fgc_stops.sort_values(by='stop_name',inplace=True)
geo_df_fgc_stops.rename(columns={'stop_id': 'id','stop_name': 'name'}, inplace=True)
geo_df_fgc_stops['stop_type'] = 'FGC'
geo_df_fgc_stops= geo_df_fgc_stops[['id','name','linia','stop_type','geometry']]
geo_df_fgc_stops['id'] = 'F' + '-' + geo_df_fgc_stops['linia'] + '-' + geo_df_fgc_stops['id']
geo_df_fgc_stops

,id,name,linia,stop_type,geometry
0,F-L8-AL,Almeda,L8,FGC,POINT (2.08525 41.35307)
11,F-L8-LH,Av. Carrilet,L8,FGC,POINT (2.10265 41.35852)
33,F-L7-TB,Av. Tibidabo,L7,FGC,POINT (2.13716 41.40965)
37,F-S1-VL,Baixador de Vallvidrera,S1,FGC,POINT (2.09694 41.42009)
18,F-S1-PC,Catalunya,S1,FGC,POINT (2.16972 41.38771)
17,F-L7-PC,Catalunya,L7,FGC,POINT (2.16972 41.38771)
16,F-L6-PC,Catalunya,L6,FGC,POINT (2.16972 41.38771)
25,F-S1-PR,Diagonal,S1,FGC,POINT (2.16044 41.39568)
24,F-L7-PR,Diagonal,L7,FGC,POINT (2.16044 41.39568)
23,F-L6-PR,Diagonal,L6,FGC,POINT (2.16044 41.39568)


In [8]:
geo_df_fgc_stops.to_csv('Nodes/FGC.csv', index=False)